# Lean-19b : Borne de Chernoff pour le BSC -- extension de Lean-19

**Navigation** : [<< Lean-19 Hoeffding](Lean-19-Detection-MIMO-Index-Python.ipynb) · [Lean-19c Bernstein](Lean-19c-Borne-Bernstein-Python.ipynb) · [Lean-19d Bayésien](Lean-19d-Inference-Bayesian-Beta-Binomial-Python.ipynb) · [Sommaire série](README.md)
**Navigation** : [<< Lean-19 Hoeffding](Lean-19-Detection-MIMO-Index-Python.ipynb) · [Lean-19c Bernstein](Lean-19c-Borne-Bernstein-Python.ipynb) · [Lean-19d Bayésien](Lean-19d-Inference-Bayesian-Beta-Binomial-Python.ipynb) · [Sommaire série](README.md)
Carnet d'**extension methodologique** de Lean-19 (Hoeffding) : derivation de la borne de
Chernoff pour le canal binaire symetrique (BSC), comparaison rigoureuse avec Hoeffding
(distribution-free) et Bernstein (intermediaire) sur 20 instances BSC.

**Enonce** : Pour X = somme de N variables iid Bernoulli(p), on derive la borne de
Chernoff en exploitant la fonction generatrice des moments (mgf) exacte de la Bernoulli :
```
M_X(t) = E[e^(tX)] = (1 - p) + p * e^t
```
La borne de Chernoff (approx KL quadratique) est :
```
P(|p_hat - p| >= eps) <= 2 * exp(-N * eps^2 / (2 p (1-p)))
```
La borne de Hoeffding (distribution-free) suppose le pire cas p=1/2 :
```
P(|p_hat - p| >= eps) <= 2 * exp(-2 N eps^2)
```

**Pourquoi ce carnet est un sous-carnet de Lean-19** : Lean-19 a etabli empiriquement que
la detection MIMO tient avec la borne de Hoeffding sur 20 instances. Lean-19b montre
que la **borne de Chernoff est plus tight** pour la distribution Bernoulli, et fournit
l'outil de reference pour la detection MIMO quand la distribution est connue.


In [1]:
import numpy as np

# Graine pour reproductibilite (meme seed que Lean-19)
rng = np.random.default_rng(seed=42)
print("Lean-19b seed=42 (rng numpy PCG64, continuation de Lean-19)")


Lean-19b seed=42 (rng numpy PCG64, continuation de Lean-19)


In [2]:
def hoeffding_eps(N, p, eps_p=0.05):
    # Borne de Hoeffding (two-sided, distribution-free, range [0,1]).
    # Forme standard : P(|p_hat - p| >= eps) <= 2 * exp(-2 * N * eps^2).
    return float(np.sqrt(np.log(2.0 / eps_p) / (2.0 * N)))

def chernoff_eps(N, p, eps_p=0.05):
    if p == 0 or p == 1:
        return 0.0
    return float(np.sqrt(2.0 * np.log(2.0 / eps_p) * p * (1.0 - p) / N))

def bernstein_eps(N, p, eps_p=0.05):
    # Borne de Bernstein (two-sided, range [0,1]).
    # Forme standard : P(|p_hat - p| >= eps) <= 2 * exp(-N * eps^2 / (2*(sigma^2 + eps/3))).
    # Solution fermee : eps = ((2/3)*L + sqrt((2/3)^2 L^2 + 8*N*L*sigma^2)) / (2*N),
    # avec L = ln(2/eps_p) et sigma^2 = p*(1-p).
    if p == 0 or p == 1:
        return hoeffding_eps(N, p, eps_p)
    L = np.log(2.0 / eps_p)
    sigma2 = p * (1.0 - p)
    disc = (2.0 / 3.0 * L) ** 2 + 8.0 * N * L * sigma2
    return float((2.0 / 3.0 * L + np.sqrt(disc)) / (2.0 * N))

# 12 instances de reference (meme seed que Lean-19)
instances_ref = []
for i in range(12):
    M = 4 + i
    t = 3 + (i % 3)
    p = 0.10 + 0.05 * i
    n_runs = 1000
    N = n_runs * t
    n_flips = 0
    for _ in range(n_runs):
        etat = rng.integers(0, M)
        for _ in range(t):
            if rng.random() < p:
                etat = (etat + 1) % M
                n_flips += 1
    p_hat = n_flips / N
    eps_H = hoeffding_eps(N, p)
    eps_C = chernoff_eps(N, p)
    eps_B = bernstein_eps(N, p)
    instances_ref.append((M, t, p, N, p_hat, eps_H, eps_C, eps_B))

print("=== Cellule 1 : mgf Bernoulli et borne de Chernoff ===")
print("Pour X ~ Bernoulli(p), la fonction generatrice des moments est :")
print("  M_X(t) = E[e^(tX)] = (1 - p) + p * e^t")
print()
print("Borne de Chernoff (two-sided, approx KL) :")
print("  P(|p_hat - p| >= eps) <= 2 * exp(-N * eps^2 / (2 p (1-p)))")
print()
print("Borne de Hoeffding (two-sided, distribution-free) :")
print("  P(|p_hat - p| >= eps) <= 2 * exp(-2 N eps^2)")
print()
print("Pour un meme niveau de confiance 1 - eps_p = 0.95 :")
print(f"{'M':>4} {'t':>3} {'p':>6} {'N':>6} {'p_hat':>7} {'hoeff(H)':>9} {'chern(C)':>9} {'bernst(B)':>9}")
for M, t, p, N, p_hat, eps_H, eps_C, eps_B in instances_ref:
    print(f"{M:>4} {t:>3} {p:>6.3f} {N:>6} {p_hat:>7.4f} {eps_H:>9.5f} {eps_C:>9.5f} {eps_B:>9.5f}")


=== Cellule 1 : mgf Bernoulli et borne de Chernoff ===
Pour X ~ Bernoulli(p), la fonction generatrice des moments est :
  M_X(t) = E[e^(tX)] = (1 - p) + p * e^t

Borne de Chernoff (two-sided, approx KL) :
  P(|p_hat - p| >= eps) <= 2 * exp(-N * eps^2 / (2 p (1-p)))

Borne de Hoeffding (two-sided, distribution-free) :
  P(|p_hat - p| >= eps) <= 2 * exp(-2 N eps^2)

Pour un meme niveau de confiance 1 - eps_p = 0.95 :
   M   t      p      N   p_hat  hoeff(H)  chern(C) bernst(B)
   4   3  0.100   3000  0.1007   0.02480   0.01488   0.01529
   5   4  0.150   4000  0.1460   0.02147   0.01534   0.01565
   6   5  0.200   5000  0.2032   0.01921   0.01537   0.01561
   7   3  0.250   3000  0.2423   0.02480   0.02147   0.02189
   8   4  0.300   4000  0.2945   0.02147   0.01968   0.01999
   9   5  0.350   5000  0.3488   0.01921   0.01832   0.01857
  10   3  0.400   3000  0.3857   0.02480   0.02429   0.02471
  11   4  0.450   4000  0.4477   0.02147   0.02137   0.02168
  12   5  0.500   5000  0.4866  

## Mesure 2 : extension sur 8 nouvelles instances

Les 12 instances precedentes varient `M` (4 a 15) et `t` (3 a 5) avec `p` lineaire de 0.10 a 0.65.
Pour verifier que le gain Chernoff tient au-dela, on elargit a 8 nouvelles instances avec `M`
(6 a 20), `p` plus eleve (0.20 a 0.76) et `n_runs` double (2000), ce qui reduit tous les
ecarts en `1/sqrt(N)`.


In [3]:
# 8 nouvelles instances : plage elargie (M, p) et n_runs double
instances_new = []
for i in range(8):
    M = 6 + 2 * i
    t = 4 + (i % 2)
    p = 0.20 + 0.08 * i
    n_runs = 2000
    N = n_runs * t
    n_flips = 0
    for _ in range(n_runs):
        etat = rng.integers(0, M)
        for _ in range(t):
            if rng.random() < p:
                etat = (etat + 1) % M
                n_flips += 1
    p_hat = n_flips / N
    eps_H = hoeffding_eps(N, p)
    eps_C = chernoff_eps(N, p)
    eps_B = bernstein_eps(N, p)
    instances_new.append((M, t, p, N, p_hat, eps_H, eps_C, eps_B))

print("=== Cellule 2 : Hoeffding vs Chernoff sur 8 nouvelles instances ===")
print("Meme methode, plage elargie (n_runs=2000) :")
print(f"{'M':>4} {'t':>3} {'p':>6} {'N':>6} {'p_hat':>7} {'hoeff(H)':>9} {'chern(C)':>9} {'bernst(B)':>9}")
for M, t, p, N, p_hat, eps_H, eps_C, eps_B in instances_new:
    print(f"{M:>4} {t:>3} {p:>6.3f} {N:>6} {p_hat:>7.4f} {eps_H:>9.5f} {eps_C:>9.5f} {eps_B:>9.5f}")


=== Cellule 2 : Hoeffding vs Chernoff sur 8 nouvelles instances ===
Meme methode, plage elargie (n_runs=2000) :
   M   t      p      N   p_hat  hoeff(H)  chern(C) bernst(B)
   6   4  0.200   8000  0.1998   0.01518   0.01215   0.01230
   8   5  0.280  10000  0.2774   0.01358   0.01220   0.01232
  10   4  0.360   8000  0.3545   0.01518   0.01458   0.01473
  12   5  0.440  10000  0.4404   0.01358   0.01348   0.01361
  14   4  0.520   8000  0.5104   0.01518   0.01517   0.01533
  16   5  0.600  10000  0.6074   0.01358   0.01331   0.01343
  18   4  0.680   8000  0.6785   0.01518   0.01417   0.01432
  20   5  0.760  10000  0.7641   0.01358   0.01160   0.01172


## Mesure 3 : confrontation Chernoff vs Hoeffding globale

Agregation des 20 instances : on compare les ecarts `eps_H` (Hoeffding), `eps_C` (Chernoff)
et `eps_B` (Bernstein) au niveau de confiance 95 %. Le plus petit eps = la borne la plus
stricte = la plus **tight**.


In [4]:
all_inst = instances_ref + instances_new
hoeffs = [h for *_, h, _, _ in all_inst]
cherns = [c for *_, _, c, _ in all_inst]
berns = [b for *_, _, _, b in all_inst]

print("=== Cellule 3 : Synthese Chernoff vs Hoeffding sur 20 instances ===")
print(f"Ecart Hoeffding : min={min(hoeffs):.5f}, max={max(hoeffs):.5f}")
print(f"Ecart Chernoff  : min={min(cherns):.5f}, max={max(cherns):.5f}")
print(f"Ecart Bernstein : min={min(berns):.5f}, max={max(berns):.5f}")
print()
pC = sum(1 for h, c in zip(hoeffs, cherns) if c < h)
pH = len(hoeffs) - pC
print("Comparaison hoeff vs chern :")
print(f"  - Chernoff plus tight (eps_C < eps_H) : {pC}/20")
print(f"  - Hoeffding plus tight (eps_H < eps_C) : {pH}/20")
print()
print("Resultat : Chernoff est plus tight que Hoeffding pour Bernoulli car elle exploite")
print("la vraie variance p(1-p) au lieu du majorant Hoeffding (1/4) qui suppose le pire cas.")
print("Le gain est maximal pour p proche de 0 ou 1 (ou p(1-p) << 1/4).")
print("Bernstein est intermediaire : exploite la variance p(1-p) mais reste conservative pour t grand.")
print("Chernoff est la borne de choix quand on connait la distribution exacte.")


=== Cellule 3 : Synthese Chernoff vs Hoeffding sur 20 instances ===
Ecart Hoeffding : min=0.01358, max=0.02480
Ecart Chernoff  : min=0.01160, max=0.02467
Ecart Bernstein : min=0.01172, max=0.02508

Comparaison hoeff vs chern :
  - Chernoff plus tight (eps_C < eps_H) : 19/20
  - Hoeffding plus tight (eps_H < eps_C) : 1/20

Resultat : Chernoff est plus tight que Hoeffding pour Bernoulli car elle exploite
la vraie variance p(1-p) au lieu du majorant Hoeffding (1/4) qui suppose le pire cas.
Le gain est maximal pour p proche de 0 ou 1 (ou p(1-p) << 1/4).
Bernstein est intermediaire : exploite la variance p(1-p) mais reste conservative pour t grand.
Chernoff est la borne de choix quand on connait la distribution exacte.


## Interpretation

**Resultat** : sur 19/20 instances BSC, la borne de Chernoff est plus tight que Hoeffding
(eps_C < eps_H) ; la 20e (p=0.50) donne des ecarts numeriquement egaux (eps_C = eps_H),
ce qui est attendu theoriquement (a p=1/2, 2*p*(1-p) = 1/2, l'exposant
-N*eps^2 / (2*p*(1-p)) devient -2*N*eps^2 et Chernoff se reduit exactement
a la forme Hoeffding). Le gain vient de ce que Chernoff exploite la **vraie variance** `p(1-p)`
de la Bernoulli, alors que Hoeffding utilise le majorant grossier `1/4` (pire cas p=1/2).
Quand p est proche de 0 ou de 1, la variance reelle est beaucoup plus petite que 1/4, et
le gain de Chernoff est maximal (jusqu'a 1.67x plus tight pour p=0.1, N=3000 :
eps_H = 0.02480 vs eps_C = 0.01488).

**Bernstein** est une borne intermediaire : elle exploite la variance `p(1-p)` comme
Chernoff, mais reste conservative pour les ecarts `t` grands (terme `t/3` au denominateur).
Elle est particulierement adaptee quand on veut une borne de queue bilaterale qui ne suppose
pas la borne multiplicative exacte de Chernoff.

**Implication pour la detection MIMO** : Lean-19 etablissait que la detection tient avec
Hoeffding (borne conservative, distribution-free). Lean-19b montre qu'on peut faire mieux
avec Chernoff quand on connait la distribution exacte (Bernoulli dans le cas BSC). Pour
le cas MIMO general (non-BSC), Bernstein sera probablement le bon compromis.

**Limites** : (a) l'approximation KL quadratique n'est valide que pour delta petit ; pour
delta grand, la borne exacte est `exp(-N * KL(p+delta || p))` avec KL la divergence de
Kullback-Leibler. (b) Pour les distributions non-Bernoulli, Chernoff ne s'applique pas
directement et Bernstein devient preferable. (c) Le gain de Chernoff depend du modele ;
pour p=1/2, Chernoff et Hoeffding sont numeriquement egaux (le ratio theorique 2*sqrt(p(1-p)) = 1 a p=1/2, et les eps mesures sont identiques a 0.01921 sur 5/20 cas = 0.50, ligne M=12, cf. table ci-dessus).


## Exercices

### Exercice 1 : fonction generatrice des moments de la Bernoulli

Implementer `mgf_bernoulli(t, p)` qui retourne la mgf d'une Bernoulli(p) evaluee en `t` :
`M(t) = (1 - p) + p * e^t`. Verifier que `M(0) = 1` et `M'(0) = p` (esperance).


In [5]:
def mgf_bernoulli(t, p):
    # TODO etudiant : retourner (1 - p) + p * e^t
    pass

# Test : M(0) = 1, M'(0) = p
m0 = mgf_bernoulli(0, 0.3)
if m0 is not None:
    print(f"M(0, p=0.3) = {m0:.6f} (attendu 1.0)")
    print(f"M(0.5, p=0.3) = {mgf_bernoulli(0.5, 0.3):.6f}")
else:
    print("Exercice 1 a completer")


Exercice 1 a completer


### Exercice 2 : borne de Chernoff (approx KL) pour un ecart donne

Implementer `chernoff_eps(N, p, eps_p=0.05)` qui retourne l'ecart `eps` tel que
`P(|p_hat - p| >= eps) <= eps_p` sous Chernoff (approx KL quadratique). Verifier que
pour `N=1000, p=0.1, eps_p=0.05`, l'ecart est proche de `0.0258` (= sqrt(2*ln(40)*0.09/1000), cf. table ci-dessus).


In [6]:
def chernoff_eps_exo(N, p, eps_p=0.05):
    # TODO etudiant : retourner sqrt(2 * log(2/eps_p) * p * (1-p) / N)
    return None  # None -> TODO etudiant

reponse = chernoff_eps_exo(1000, 0.1)
if reponse is not None:
    print(f"Chernoff : eps = {reponse:.5f} (attendu ~0.0258)")
else:
    print("Exercice 2 a completer")


Exercice 2 a completer


### Exercice 3 : comparer Chernoff et Hoeffding sur une plage parametrique

Tracer (ou afficher en table) le ratio `eps_H / eps_C` en fonction de `p` pour `N=5000`
et `eps_p=0.05`. Observer que le gain de Chernoff est maximal pour p proche de 0 ou 1,
et minimal (egal a 1.0 en ratio `eps_H / eps_C`) pour p=1/2. Le ratio theorique derive des deux formes : `eps_C / eps_H = sqrt(2 * p * (1-p) * 2) = 2*sqrt(p*(1-p))`. A p=0.1, le ratio vaut `0.6` (Chernoff ~1.67x plus tight, cf. table p=0.1 N=3000) ; a p=0.5, le ratio vaut `1.0` (egalite numerique, cf. ligne M=12) ; le gain est asymmetrique : Chernoff se reduit a Hoeffding au pire cas, et devient beaucoup plus tight quand p s'eloigne de 1/2.


In [7]:
def ratio_chernoff_hoeffding(N, p, eps_p=0.05):
    # TODO etudiant : retourner eps_H / eps_C
    pass

if ratio_chernoff_hoeffding is not None:
    print("Ratio eps_H / eps_C pour N=5000, eps_p=0.05 :")
    for p_val in [0.05, 0.1, 0.2, 0.3, 0.4, 0.5]:
        r = ratio_chernoff_hoeffding(5000, p_val)
        if r is not None:
            print(f"  p = {p_val:.2f} : ratio = {r:.3f}")
else:
    print("Exercice 3 a completer")


Ratio eps_H / eps_C pour N=5000, eps_p=0.05 :


## Conclusion et references

Lean-19b etablit que la borne de Chernoff est plus tight que Hoeffding pour la distribution
Bernoulli, sur 19/20 instances BSC (egalite numerique a p=0.50 sur la 20e). Le gain provient de l'exploitation de la variance reelle
`p(1-p)` au lieu du majorant Hoeffding `1/4`. Bernstein est une borne intermediaire, plus
polyvalente pour les distributions non-bornees.

Quand la PR #20371 mergera et que les carnets MIMO post-capstone executeront la detection
sur le vrai canal MIMO du lac `mimo_lean`, Chernoff et Bernstein deviendront les outils de
choix pour des bornes de queue informatives.

**References** :

- `Lean-19-Detection-MIMO-Index-Python.ipynb` -- le carnet pere, etablit la borne Hoeffding
  empiriquement sur 20 instances (20/20 sous la borne).
- `Lean-20-Capstone-Digestions-Tao-Python.ipynb` -- la capstone de la serie, qui digere les
  techniques de Terence Tao sur les flots de Ricci.
- `Lean-21c-Descente-Budget.ipynb` (futur `MIMO/MIMO-01-Descente-Budget-Lean-Python.ipynb`)
  -- la borne de budget pour la detection MIMO, qui consumera Lean-19 et Lean-19b.
- PR #20371 (en attente) -- la descente MIMO-01..04 qui consommera Lean-19b comme etalon
  de borne de queue.
